# Whisper LoRA 微调模型评估

在衡东方言验证集上评估 Whisper-base + LoRA 微调后的效果。

## 1. 环境检查

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, re, json, time, gc
import torch

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

try:
    import librosa
    print(f'librosa: {librosa.__version__}')
except ImportError:
    print('ERROR: librosa 未安装')

try:
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    import transformers
    print(f'transformers: {transformers.__version__}')
except ImportError:
    print('ERROR: transformers 未安装')

try:
    from peft import PeftModel
    import peft
    print(f'peft: {peft.__version__}')
except ImportError:
    print('ERROR: peft 未安装')

import editdistance
print(f'editdistance: OK')

print('\n--- 路径检查 ---')
MODEL_DIR = os.path.expanduser('~/Projects/Agent/local/whisper-base')
LORA_DIR = os.path.expanduser('~/Projects/Agent/local/lora_adapter')
VAL_JSONL = os.path.expanduser('~/Desktop/hengdong_asr_trainset/manifests/val.jsonl')
for name, path in [('Whisper-base', MODEL_DIR), ('LoRA adapter', LORA_DIR), ('验证集', VAL_JSONL)]:
    print(f'  {name}: {"OK" if os.path.exists(path) else "NOT FOUND"} - {path}')

## 2. 工具函数

In [ ]:
_PUNCT = re.compile(
    r'[\s\.,!?;:\-()\[\]{}\u3000\uff0c\u3002\uff01\uff1f\u3001\uff1b\uff1a'
    r'\u201c\u201d\u2018\u2019\uff08\uff09\u3010\u3011\u300a\u300b\u2026\u2014]'
)

def norm_punct(text):
    """去除标点符号（用于 CER 计算）"""
    return _PUNCT.sub('', text)

def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for c1 in s1:
        curr = [prev[0] + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def free_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

print('工具函数就绪')

## 3. 数据加载

In [ ]:
DATA_ROOT = os.path.expanduser('~/Desktop/hengdong_asr_trainset')
VAL_JSONL = os.path.join(DATA_ROOT, 'manifests/val.jsonl')

samples = []
with open(VAL_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        audio = s.get('audio_filepath') or s.get('source')
        if audio and os.path.exists(audio):
            samples.append(s)

print(f'验证集: {len(samples)} 条有效样本')
print(f'\n前3条:')
for s in samples[:3]:
    audio = s.get('audio_filepath') or s.get('source')
    text = s.get('text') or s.get('target')
    print(f'  {os.path.basename(audio)} -> {text}')

## 4. 加载 Whisper-base + LoRA

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
import librosa

MODEL_DIR = os.path.expanduser('~/Projects/Agent/local/whisper-base')
LORA_DIR = os.path.expanduser('~/Projects/Agent/local/lora_adapter')

print('加载 Whisper-base 基座模型...')
start = time.time()

processor = WhisperProcessor.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    language='chinese',
    task='transcribe',
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)
model = model.to(device)

print('应用 LoRA 权重...')
model = PeftModel.from_pretrained(model, LORA_DIR)
model.eval()

# forced_decoder_ids 强制中文解码
forced_decoder_ids = processor.get_decoder_prompt_ids(language='chinese', task='transcribe')

print(f'模型加载完成, 耗时: {time.time() - start:.1f}s')
model.print_trainable_parameters()

## 5. 评估函数

In [ ]:
def eval_whisper_lora(model, processor, samples, label, forced_decoder_ids):
    """评估 Whisper + LoRA 微调模型"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio = s.get('audio_filepath') or s.get('source')
            expected_raw = s.get('text') or s.get('target')
            if not audio or not os.path.exists(audio):
                continue
            
            try:
                array, sr = librosa.load(audio, sr=16000)
                input_features = processor.feature_extractor(
                    raw_speech=array,
                    sampling_rate=sr,
                    return_tensors='pt',
                ).input_features.to(device)
                
                generated_ids = model.generate(
                    input_features=input_features,
                    forced_decoder_ids=forced_decoder_ids,
                )
                pred_raw = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            except Exception as e:
                pred_raw = ''
                print(f'Error at {i}: {e}')
            
            expected = norm_punct(expected_raw)
            pred = norm_punct(pred_raw)
            dist = levenshtein(expected, pred)
            ref_len = max(len(expected), 1)
            cer = dist / ref_len
            total_cer += dist
            total_chars += ref_len
            if expected == pred:
                exact += 1
            
            results.append({
                'id': i,
                'expected_raw': expected_raw,
                'predicted_raw': pred_raw,
                'expected': expected,
                'predicted': pred,
                'cer': cer
            })
            
            if (i + 1) % 50 == 0:
                print(f'    {label}: {i+1}/{len(samples)} | CER={total_cer/total_chars:.2%} | exact={exact}')
    
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    return {
        'name': label,
        'results': results,
        'cer': cer,
        'exact': exact,
        'total': len(results),
        'time': elapsed
    }

print('评估函数就绪')

## 6. 评估 Whisper LoRA

In [ ]:
print('评估 Whisper-base + LoRA 微调...')
lora_res = eval_whisper_lora(model, processor, samples, 'Whisper-LoRA', forced_decoder_ids)

print(f'\n完成! CER={lora_res["cer"]:.2%} | exact={lora_res["exact"]}/{lora_res["total"]} | 耗时={lora_res["time"]:.1f}s')

## 7. 结果汇总

In [ ]:
print('=' * 60)
print('Whisper-base + LoRA 微调模型评估结果 (验证集, 标点归一化)')
print('=' * 60)
print(f'CER:           {lora_res["cer"]:.2%}')
print(f'精确匹配率:     {lora_res["exact"]}/{lora_res["total"]} ({lora_res["exact"]/lora_res["total"]:.1%})')
print(f'推理耗时:       {lora_res["time"]:.1f}s')
print(f'速度:          {lora_res["total"]/lora_res["time"]:.1f} 条/秒')
print('=' * 60)

## 8. 保存结果

In [ ]:
REPORT_DIR = os.path.expanduser('~/Projects/Agent/local')

save_path = os.path.join(REPORT_DIR, 'whisper_lora_result.json')
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump({
        'model': 'Whisper-base + LoRA (微调)',
        'cer': round(lora_res['cer'], 4),
        'exact_match': lora_res['exact'],
        'total': lora_res['total'],
        'exact_rate': round(lora_res['exact'] / max(lora_res['total'], 1), 4),
        'time': round(lora_res['time'], 1),
        'speed': round(lora_res['total'] / lora_res['time'], 1),
        'samples': [
            {
                'id': r['id'],
                'expected': r['expected'],
                'expected_raw': r['expected_raw'],
                'predicted': r['predicted'],
                'predicted_raw': r['predicted_raw'],
                'cer': round(r['cer'], 4)
            }
            for r in lora_res['results']
        ]
    }, f, ensure_ascii=False, indent=2)

print(f'结果已保存: {save_path}')

## 9. 清理内存

In [ ]:
del model, processor
free_memory()
print('内存已清理')